# Problem 043:  Sub-string Divisibility

> The number, $1406357289$, is a $0$ to $9$ pandigital number because it is made up of each of the digits $0$ to $9$ in some order, but it also has a rather interesting sub-string divisibility property.
>
> Let $d_1$ be the $1$st digit, $d_2$ be the $2$nd digit, and so on. In this way, we note the following:
> - $d_2d_3d_4=406$ is divisible by $2$
> - $d_3d_4d_5=063$ is divisible by $3$
> - $d_4d_5d_6=635$ is divisible by $5$
> - $d_5d_6d_7=357$ is divisible by $7$
> - $d_6d_7d_8=572$ is divisible by $11$
> - $d_7d_8d_9=728$ is divisible by $13$
> - $d_8d_9d_{10}=289$ is divisible by $17$
>
> Find the sum of all $0$ to $9$ pandigital numbers with this property.

## Naive Solution $\mathcal O(N!)$

The first thing that can be done is just loop over all the permuations and check if the property holds.

I will generalize the problem as follows.  Let the search be for palindromes of any set of digits.  Also, let the divisibility property be for set of three consecutive digits but for an arbitrary set of factors.

There are $\sim N!$ palindromes of a string, leading to the above time scaling (disregarding repeated digits and the possibility that there cannot be any leading $0$'s).

In [54]:
%%time

from itertools import permutations

def p043_naive(digits: list[int], factors: list[int]) -> int:
    val = 0

    d = sum([[i] * n for i, n in enumerate(digits)], [])
    xp = ()
    for x in permutations(d):
        if x != xp:
            y = 10 * x[0] + x[1]
            for a, f in zip(x[2:], factors):
                y = (10 * y + a) % 1_000
                if y % f:
                    break
            else:
                val += int(''.join(map(str, x)))
                #print(x)
        xp = x
    
    return val

digits = [1] * 10
factors = [1, 2, 3, 5, 7, 11, 13, 17]
ans = p043_naive(digits, factors)

print(ans)

16695334890
CPU times: user 2.77 s, sys: 0 ns, total: 2.77 s
Wall time: 2.76 s


## Dynamic programming Solution $\mathcal O\left( e^{\alpha N} \right)$

The above code obviously treads a lot of ground that is unnecessary.  For example, if the fourth digits in the permutation is odd, then the search through permutations could be pruned instead of looping over the next $6!$ permutations and discovering that they all have the same odd digit in the fourth position.

A much more efficient approach is to used dynamic programming by growing the number with the desired properties and pruning whenever it is impossible to have all the properties.  To this end, it is easier to start at the most significant digit and work your way down.  The calculation that is required for the $n^\mbox{th}$ digit is to find the next multiple of the required factor that is greater than $100 \times d_{n-2} + 10 \times d_{n-1}$.  This digit can also be increased by that same factor until it goes over $10$.  The available digits are kept track of and the growth continues whenever the next digit available is added.

Avoiding all the redundant calculations in the first algorithm reduces the time of calculation by a factor of $100$.  As far as the time scaling, I put it as exponential in the number of digits in the palindrome.  The logic here is that if digits $d_{n-2}d_{n-1}d_{}n$ need to be divisible by $f_n$, then there are about $\lceil 10 / f_n \rceil$ cases that need to be considered.  If the factors are greater than $10$, then there is only at most one digit that is possible in each position, and the $\alpha$ will be small, otherwise the scaling could get bad if there are a lot of factors of $2$, for example.

In [53]:
%%time

def p043_dp(digits: list[int], factors: list[int]) -> int:
    val = 0

    nfac = len(factors) - 1
    
    d = [None] * sum(digits)

    for i in range(100):
        dnow = i // 10
        if digits[dnow]:
            digits[dnow] -= 1
        else:
            continue
        d[0] = dnow
        
        dnow = i % 10
        if digits[dnow]:
            digits[dnow] -= 1
        else:
            digits[d[0]] += 1
            continue
        d[1] = dnow

        n = 0
        d[2] = 0
        dnow = 0
        while True:
            if digits[dnow] == 0:
                dnow += factors[n]
            elif n == nfac:
                val += int(''.join(map(str, d)))
                #print(d)
                dnow += factors[n]
            else:
                digits[dnow] -= 1
                n += 1
                y = 10 * (10 * d[n] + d[n+1])
                dnow = ((y - 1) // factors[n] + 1) * factors[n] - y
            while n > 0 and dnow > 9:
                n -= 1
                dnow = d[n+2]
                digits[dnow] += 1
                dnow += factors[n]
            if n == 0 and dnow > 9:
                break
            d[n+2] = dnow
        digits[d[0]] += 1
        digits[d[1]] += 1
    
    return val

digits = [1] * 10
factors = [1, 2, 3, 5, 7, 11, 13, 17]
ans = p043_dp(digits, factors)

print(ans)

16695334890
CPU times: user 24.1 ms, sys: 1.1 ms, total: 25.2 ms
Wall time: 23.8 ms


It is interesting to note that there are some shortcuts that could be used in the specific problem that are difficult to generalize.  For example, it is clear that $d_6 = 5$ due to the fact that $d_4d_5d_6$ is divisible by $5$ and $d_6d_7d_8$ is divisible by $11$.  I went for generalizing the code as opposed to optimizing the specific case.